In [ ]:
import numpy as np
import tifffile as tiff
from matplotlib import pyplot as plt
import cv2 as cv
import scipy as sp
import trad_cv_method as tradcv
from scipy.ndimage import shift
from skimage.registration import phase_cross_correlation
from skimage.transform import AffineTransform, warp
import matplotlib as mpl
from skimage import data, img_as_float
from skimage import exposure
import skimage
from PIL import Image
import os
import imageio.v2 as imageio
import glob
import seaborn as sns
import pandas as pd
#import eraser_tool

# **IMAGE ANALYSIS OF BRAIN ENDOTHELIAL Ca2+ SIGNALS**

This notebook was created after we began collecting data from our new microscope. Uses the traditional CV methods, but this notebook has less notes and comments than the tutorial notebook. It also accomodates for an eraser tool to erase noise and donuts from the active site masks.

Assumed folder structure:
```
12favg_5xfad_barium_data/
├──12favg_combined/
|   ├── Mouse1/
|   |   ├── Calcium channel tiffs...
|   |   └── Plasma/
|   |       └── Plasma channel tiffs...
|   └── Mouse2/ ... etc
|
├──12favg_stmaps/
|   ├── Mouse1_stmap.png etc...
|   └── STMap_arrays/
|       └── Mouse1_stmaparray.npy etc...
|
├──erased_masks/
|   └── Mouse1_fov1_prebarium.npy (if needed) etc...
|
├──imaging_fov/
|   └── Mouse1_fov1_prebarium_fov.npy etc...
|
└──raw_masks/
    └── Mouse1_fov1_prebarium_rawmasks.npy etc...
```
If you change the names of the folders for different experiments, datasets, etc... , you will have to change the directories in the cells below to the new ones. <br>
In cell below, change `filepath` to path of 12 frame averaged video file (calcium/green channel).<br>
Make sure to change `sample_id` every time!!<br>
To avoid any bugs, `sample_id` **MUST** be in the following format (and make sure it is like this everywhere, including in data tables, etc. for consistency)!!!
* **{Experimental Group}** *space* **{Animal #}** *space* **{Sex (M or F)}** *space* **{prebarium or postbarium (or whatever treatment group you are using)}** *space* **{FOV#}**
    * **Example: Cdh10 99 M prebarium FOV1**
    * Experimental Group = 6xFAD ctrl, 6xFAD tg+, Cdh10, etc (this is not case sensitive if you are on a Mac)
    * Must put a space between each element in the sample ID (i.e. Must be Cdh10 99 M prebarium FOV1 and **NOT** Cdh10 99M prebarium FOV1)
    * Do not put space between "FOV" and the FOV number (i.e. must be 'FOV1' or 'FOV2' and **NEVER** 'FOV 1' etc)
  
Also, set `project_data_dir` to the location of the parent folder containing all data (frame averaged tiffs, ST Maps, etc...) for each project. It should be the same as `project_data_dir` from the save_frame_averaged_tiffs notebook. For example, in the 5xFAD barium project, `project_data_dir` would be "/Volumes/Shares/NVR/Folder1 Lab Personnel/Joshua Li/12favg_data/".

Generally speaking, you can just run cells below sequentially until it says to **Pause**

In [ ]:
# CHANGE filepath FOR EACH VIDEO FILE
filepath = ''
# MAKE SURE TO CHANGE sample_id. MANUALLY SET EACH TIME
sample_id = ''
project_data_dir = ''

video = tiff.imread(filepath)

filename = filepath.split("/")[-1].split('.')[0]
directory,_ = os.path.split(filepath)

directory

In [ ]:
denoised_video = tradcv.denoiseVideo(video[:], avg_size=12)
xy_corrected_vid, xcb, ycb = tradcv.correct_motion(denoised_video[:].astype(np.uint16))


In [ ]:
plt.imshow(np.mean(xy_corrected_vid, axis=0), vmax=500, cmap='gray')
plt.show()
save_path = f"/Users/garfinkeljb/Downloads/12favg_{sample_id}_xycorr.tif"
tiff.imwrite(save_path, xy_corrected_vid.astype(np.uint16))

In [ ]:
sd_video = tradcv.SD(xy_corrected_vid, span=30)
plt.imshow(np.max(sd_video, axis=0), cmap='plasma')
plt.show()

In [ ]:
idb_video = tradcv.idB(sd_video)
plt.imshow(np.max(idb_video, axis=0), cmap='binary', norm=mpl.colors.Normalize(vmin=0,vmax=20))
plt.show()

In [ ]:
threshold = 10 # Leave at 10
#threshold = 8
print(xcb, ycb)
thresh_array = tradcv.multiThreshold(idb_video[:], threshold, xcb,ycb)
plt.imshow(thresh_array[:,:], cmap='binary')
plt.show()

`particle_size` has been changed from 25 to 20 because of the larger FOV compared to the old data. Sometimes, 20 is still too high and you may want to go smaller (lowest I've gone is 5) if there are obvious active sites being missed. Any incidental noise particles being picked up can be erased using the eraser tool.

In [ ]:
#thresh_array[:100,:50] = 0
filtered_image, activeSite, labels, stats, boundingCoords = tradcv.particleFilter(thresh_array, particle_size=20) # 20 for new microscope instead of 25
print(np.unique(filtered_image))
plt.imshow(filtered_image, cmap='binary')

### **Pause Here**
If you have the corresponding 12 frame averaged plasma .tiffs, run the cell below. If not, skip and go the next cell.

In [ ]:
plasma_video = tiff.imread(os.path.join(directory, "plasma", f"{filename[:-3]}Ch1.tiff"))
plasma_denoised = tradcv.denoiseVideo(plasma_video)
plasma_xy_corrected, _, _ = tradcv.correct_motion(plasma_denoised[:].astype(np.uint16))


If you have a corresponding plasma tiff, `fov_to_plot` should be set to `plasma_xy_corrected`. Otherwise, set it to `xy_corrected_vid`. Using the plasma FOV will allow for easier visualization of the active site locations, but for a publication/presentation, use the calcium FOV, per Amreen's guidance.

In [ ]:
fig = plt.figure(figsize=(8,8))
fov_to_plot = plasma_xy_corrected #plasma_xy_corrected if available, otherwise xy_corrected_vid
maxxed_fov = np.max(fov_to_plot,axis=0)
plt.imshow(maxxed_fov, cmap='gray', vmin=0, vmax=np.quantile(maxxed_fov, .99))

cmap = plt.cm.rainbow  # Start with a grayscale colormap

# Create a new colormap with alpha values
new_cmap = cmap.copy()
new_cmap.set_under('none')

plt.imshow(filtered_image, cmap=new_cmap, alpha=1, vmin=0.9)

# SAVE FUNCTIONS COMMENTED OUT
np.save(os.path.join(project_data_dir, 'raw_masks', f"{sample_id}_rawmasks.npy"), filtered_image)

np.save(os.path.join(project_data_dir, 'imaging_fov', f"{sample_id}_fov.npy"), maxxed_fov.astype(np.uint16))

plt.show()
plt.close()


### **PAUSE AGAIN**

Decide whether you will need to use the eraser tool to erase noise or donuts from the active site masks. 

Skip the below cell if you did not need to erase the active site to remove noise particles or donuts. 
If you did use the eraser tool, make sure to change `erased_mask_file` to the correct file where you have the erased mask and run the cell below

In [ ]:
erased_mask_file = '/Volumes/NVR_Local_Server/Joshua/12favg_data_5xfad_barium/erased_masks/5xfad_91m_fov1_prebarium_erased.npy'
erased_activesite_map = np.load(erased_mask_file)
erased_activesite_map[np.isnan(erased_activesite_map)] = 0
erased_activesite_map, activeSite, labels, stats, boundingCoords = tradcv.particleFilter(erased_activesite_map, particle_size=20) # default particle_size = 25
filtered_image = erased_activesite_map

print(np.unique(erased_activesite_map))
plt.imshow(erased_activesite_map, cmap='binary')



In [ ]:
title = sample_id

if np.sum(filtered_image) == 0: # SAVE BLANK DUMMY STMAP ARRAY IF THERE ARE NO ACTIVE SITES
    np.save(os.path.join(project_data_dir, f'12favg_stmaps/STMap_arrays/{title}_stmaparray.npy'), np.array([[0],[0]]))


STMap = tradcv.buildSTMap(xy_corrected_vid, boundingCoords, filtered_image, mapType='zscr', # mapType = 'zscr' or 'raw'
                                                                            cutoff=2.5, # I used 5, paper used 2.5
                                                                            percentile=.35, # default = 0.35
                                                                            particle_size=13) # default = 13

frame_period = 0.033474222 * 12 # seconds per frame
frames = len(xy_corrected_vid)
minutes = frames * frame_period / 60
pixel_um = 1.08587595594912

print(np.nansum(STMap[:,:])* (frame_period*pixel_um) / minutes)

plt.figure(figsize=(10, 10))
plt.imshow(STMap[:,:], cmap='plasma', aspect=2/3, norm=mpl.colors.Normalize(vmin=0,vmax=35)) 

x_ticks = np.arange(0, STMap.shape[1], step=int(50/pixel_um))
plt.xticks(x_ticks, labels=np.round((x_ticks*pixel_um).astype(int), -1), fontsize=14)
plt.xlabel('Length (\u03BCm)', fontsize=16)

y_ticks = np.arange(0, STMap.shape[0], step=int(30/frame_period))
plt.yticks(y_ticks, labels=np.round((y_ticks*frame_period).astype(int), -1), fontsize=14)
plt.ylabel('Time (s)', fontsize=16)

plt.title(title)

# SAVE FUNCTIONS COMMENTED OUT
plt.savefig(os.path.join(project_data_dir, f'12favg_stmaps/{title}_stmap.tiff'))
np.save(os.path.join(project_data_dir, f'12favg_stmaps/STMap_arrays/{title}_stmaparray.npy'), STMap)

plt.show()
plt.close()


### **Viewing FOVs, Active Sites, and ST Maps after analysis**

The below two cells are to quickly view masks or ST Maps, after you have already generated them.

In [ ]:
#QUICKLY PLOT FOV AND MASKS

sample_id = "Cdh5 82 M prebarium FOV1"
fig = plt.figure(figsize=(8,8))
fov_to_plot = np.load(f"/Volumes/NVR/Joshua/12favg_data_5xfad_barium/imaging_fov/{sample_id}_fov.npy")

plt.imshow(fov_to_plot, cmap='gray', vmin=0, vmax=np.quantile(fov_to_plot, .99))

cmap = plt.cm.spring  # Start with a grayscale colormap

# Create a new colormap with alpha values
new_cmap = cmap.copy()
new_cmap.set_under('none')

# If you saved an erased mask you will have to change the below line to where the erased mask is saved
# Note, if the masks are blank (i.e. no active sites present at all), there will be an error
masks = np.load(f"/Volumes/NVR/Joshua/12favg_data_5xfad_barium/raw_masks/{sample_id}_rawmasks.npy")
#masks = np.load(f"/Volumes/Shares/NVR/Folder1 Lab Personnel/Joshua Li/12favg_data/erased_masks/6xfad_tg+_36M_fov1_postbarium.npy")

plt.imshow(masks, cmap=new_cmap, alpha=0.4, vmin=0.9)
plt.show()
plt.close()

In [ ]:
# QUICKLY VIEW AN ST MAP

sample_id = "6xfad tg+ 62 M prebarium FOV1"
STMap = np.load(f'/Volumes/NVR_Local_Server/Joshua/12favg_data_5xfad_barium/12favg_stmaps/STMap_arrays/{sample_id}_stmaparray.npy')

frame_period = 0.033474222 * 12 # seconds per frame
frames = len(STMap)
minutes = frames * frame_period / 60
pixel_um = 1.08587595594912

print(np.nansum(STMap[:,:])* (frame_period*pixel_um) / minutes)

plt.figure(figsize=(10, 10))
plt.imshow(STMap[:,:], cmap='plasma', aspect=2/3, norm=mpl.colors.Normalize(vmin=0,vmax=35)) 

x_ticks = np.arange(0, STMap.shape[1], step=int(50/pixel_um))
#plt.xticks(x_ticks, labels=np.round((x_ticks*pixel_um).astype(int), -1), fontsize=14)
#plt.xlabel('Length (\u03BCm)', fontsize=16)

y_ticks = np.arange(0, STMap.shape[0], step=int(30/frame_period))
#plt.yticks(y_ticks, labels=np.round((y_ticks*frame_period).astype(int), -1), fontsize=14)
#plt.ylabel('Time (s)', fontsize=16)
ax = plt.gca()
ax.set_facecolor("#ffffe4")

plt.show()
plt.close()

In [ ]:
# Control is Cdh5 82 M prebarium FOV1
# Compound: 679:761,51:78 110:210,340:367
# Proto: 105:205,208:235 280:380,230:257
# Cdh10 52 M prebarium FOV1
# Unitary: 750:800,370:400

# TG+ is 6xFAD tg+ 62 M prebarium FOV1
#Compound: 240:300,230:255
# Unitary: 50:100,195:225
# TG+ is 6xFAD tg+ 54 M prebarium FOV2 
# Proto: 540:600,165:185
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib as mpl
sample_id = 'Cdh10 52 M prebarium FOV1'
group = 'control'
type = 'unitary'
which = 0

coords = {
    'control': {
        'compound': [((679,761), (51,78)), ((110,210), (340,367))],
        'proto': [((105,205), (208,235)), ((280,380), (230,257))],
        'unitary': [((750,800), (370,400))]
    },
    '5xFAD': {
        'compound': [((240,300), (230,255))],
        'proto': [((50,100), (195,225))],
        'unitary': [((540,600),(165,185))]
    }
}    
coords = coords[group]

STMap = np.load(f'/Volumes/NVR_Local_Server/Joshua/12favg_data_5xfad_barium/12favg_stmaps/STMap_arrays/{sample_id}_stmaparray.npy')

frame_period = 0.033474222 * 12 # seconds per frame
frames = len(STMap)
minutes = frames * frame_period / 60
pixel_um = 1.08587595594912

print(np.nansum(STMap[:,:])* (frame_period*pixel_um) / minutes)

STMap_cropped = STMap[coords[type][which][0][0]:coords[type][which][0][1],
                      coords[type][which][1][0]:coords[type][which][1][1]].transpose() # CROP STMAP TO ISOLATE INDIVIDUAL ACTIVE SITE

target_shape = (30, 100) # MAX SIZE IS 30 PIXELS WIDE AND 100 PIXELS IN TIME DIMENSION
# Compute padding amounts
pad_height = target_shape[0] - STMap_cropped.shape[0]
pad_width  = target_shape[1] - STMap_cropped.shape[1]

# Make sure padding isn't negative
assert pad_height >= 0 and pad_width >= 0, "Array is larger than target size!"

# Split padding evenly (extra pixel goes on bottom/right if odd)
pad_top = pad_height // 2
pad_bottom = pad_height - pad_top
pad_left = pad_width // 2
pad_right = pad_width - pad_left

# Pad with NaNs
padded_STMap_cropped = np.pad(
    STMap_cropped,
    pad_width=((pad_top, pad_bottom), (pad_left, pad_right)),
    mode='constant',
    constant_values=np.nan
)

fig = plt.figure(figsize=(6, 5), dpi=1200)

# --- Manually positioned axes (left, bottom, width, height in 0–1 figure coords) ---
ax0 = fig.add_axes([0.1, 0.52, 0.85, 0.46])  # top plot
ax1 = fig.add_axes([0.1, 0.04, 0.85, 0.46])  # bottom plot

# --- Top plot ---
ax0.plot(np.nansum(padded_STMap_cropped, axis=0), color='black', lw=5)
ax0.set_xticks([])
ax0.tick_params('both', bottom=False, labelbottom=False, right=False, labelright=False, labelleft=False, left=False)
ax0.set_ylim(0, 400)
#ax0.set_aspect(0.1)
#ax0.yaxis.set_ticks_position('right')

# --- Bottom image ---
ax1.imshow(
    padded_STMap_cropped,
    cmap='plasma',
    norm=mpl.colors.Normalize(vmin=0, vmax=35)
)
ax1.set_yticks([])

# --- X ticks ---
#x_ticks = np.arange(0, padded_STMap_cropped.shape[1], step=(5 / frame_period))
#ax1.set_xticks(x_ticks)
#ax1.set_xticklabels(np.round((x_ticks * frame_period).astype(int), 0), fontsize=14)
ax1.tick_params('both', bottom=False, labelbottom=False, right=False, labelright=False)
#ax1.set_xlabel('Time (s)', fontsize=16)
ax1.set_aspect('auto')

for spine in ax0.spines.values():
    spine.set_visible(False)

for spine in ax1.spines.values():
    spine.set_visible(False)

plt.show()

In [ ]:
from matplotlib.lines import Line2D
# Define bar lengths (in arbitrary units)
xbar_len = 5   # e.g., 5 seconds
ybar_len = 5   # e.g., 5 pixels

# Create figure
fig, ax = plt.subplots(figsize=(2, 2), dpi=1200)
ax.axis('off')  # hide all axes

# Starting point
x0, y0 = 0.2, 0.2

# Plot scalebars
ax.add_line(Line2D([x0, x0 + xbar_len], [y0, y0], color='black', linewidth=3))  # X bar
ax.add_line(Line2D([x0, x0], [y0, y0 + ybar_len], color='black', linewidth=3))  # Y bar

# Labels
ax.text(x0 + xbar_len / 2, y0 - 0.5, f'{xbar_len} s', ha='center', va='top', fontsize=18)
ax.text(x0 - 0.5, y0 + ybar_len / 2, f'200 zscr•\u03BCm•s', ha='right', va='center', rotation=90, fontsize=18)

# Set limits to keep everything tight
ax.set_xlim(0, x0 + xbar_len + 1)
ax.set_ylim(0, y0 + ybar_len + 1)

plt.tight_layout()
plt.show()

In [ ]:
# THIS CODE WAS JUST USED TO CREATE A COLOR SCALE BAR FOR MY ABSTRACT FIGURE

import matplotlib.pyplot as plt
import numpy as np
import matplotlib as mpl

# Create a figure and axis for the colorbar only
fig, ax = plt.subplots(figsize=(4, 4), dpi=1200, )  # Increase the width to make the bar thicker
# fig.set_constrained_layout(constrained=True)
# fig.set_constrained_layout_pads(w_pad = 0, h_pad = 0, wspace=0, hspace=0)
# Define a colormap and normalization for the colorbar
cmap = mpl.cm.plasma
norm = mpl.colors.Normalize(vmin=0, vmax=35)

# Create the colorbar with the colormap and normalization
cbar = fig.colorbar(mpl.cm.ScalarMappable(cmap=cmap, norm=norm), ax=ax, aspect=10, pad=0, orientation='horizontal', fraction=1)
cbar.outline.set_linewidth(3)
# Set the ticks to [0, 35], with the upper tick labeled as '35+'
cbar.set_ticks([0, 35])
cbar.ax.tick_params(width=3)
cbar.set_ticklabels([None, None], fontsize=18)
cbar.ax.yaxis.set_label_position('left')
# Set the label for the colorbar (optional)
#cbar.set_label(f'zscr•\u03BCm•s', fontsize=18, fontweight='bold')

# Hide the axis since it's not needed for a standalone colorbar
ax.axis('off')

# Display the colorbar
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd

ipynb_files = []
for ipynb_filepath in glob.glob(os.path.join('/Users/lij49/NVR_code', '*.ipynb')):
    ipynb_filename = ipynb_filepath.split("/")[-1].split('.')[0]
    ipynb_files.append(ipynb_filename)
ipynb_df = pd.DataFrame({"ipynb_files":ipynb_files})

py_files=[]
for py_filepath in glob.glob(os.path.join('/Users/lij49/NVR_code', '*.py')):
    py_filename = py_filepath.split("/")[-1].split('.')[0]
    py_files.append(py_filename)
py_df = pd.DataFrame({"py_files":py_files})

with pd.ExcelWriter('/Users/lij49/NVR_code/code_files_notes.xlsx') as writer:
    ipynb_df.to_excel(writer, sheet_name='ipynb_files', index=False)
    py_df.to_excel(writer, sheet_name='py_files', index=False)

In [ ]:
folders = []
for path in glob.glob(os.path.join('/Volumes/Shares/NVR/Folder1 Lab Personnel/Joshua Li', '*')):
    if os.path.isdir(path):
        folder_name = path.split("/")[-1].split('.')[0]
        folders.append(folder_name)
folders_df = pd.DataFrame({"folders":folders})
folders_df.to_csv('/Volumes/Shares/NVR/Folder1 Lab Personnel/Joshua Li/shared_drive_folder_notes.csv', index=False)